In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import roc_auc_score
from tensorflow.keras.layers import Dense, Activation, BatchNormalization, LSTM, Masking, Input, GRU, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l1
from tensorflow.keras import regularizers
from sklearn.utils import shuffle
from qkeras import *
import qkeras
from tensorflow.keras.models import load_model
from qkeras.utils import model_quantize
from qkeras.utils import model_save_quantized_weights

import hls4ml

import os

import helper_functions

In [14]:
#gru layer
def grulayermodel(max_len, n_var, rec_units, ndense=[50, 10], l1_reg=0,
              l2_reg=0, rec_act='sigmoid', extra_lab='none', rec_kernel_init='VarianceScaling',
             dense_kernel_init='lecun_uniform'):
    
    hidden = x_in = Input(shape=(max_len, n_var,))
    hidden = GRU(units=rec_units,
                  recurrent_activation = rec_act,
                  kernel_initializer = rec_kernel_init, 
                  name = 'gru')(hidden)
    
    model = Model(inputs=x_in, outputs=hidden)
    
    return model

l1_reg = 0
l2_reg = 0

gru_layer_only = grulayermodel(15, 6, 120, [50, 10], l1_reg=l1_reg, l2_reg=l2_reg)
gru_layer_only.compile(optimizer='adam', loss=['categorical_crossentropy'], metrics=['accuracy']) 

gru_layer_only.load_weights("./gru_test3/gru_floating.weights.h5", skip_mismatch=True, by_name=True) 

print("nogru model")
gru_layer_only.summary()

# loading weights by layer to confirm it actually work
#for layer in gru_layer.layers:
#    weights = layer.get_weights()
#    print(layer.name, weights)

nogru model
Model: "model_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_5 (InputLayer)        [(None, 15, 6)]           0         
                                                                 
 gru (GRU)                   (None, 120)               46080     
                                                                 
Total params: 46,080
Trainable params: 46,080
Non-trainable params: 0
_________________________________________________________________


In [15]:
def nongrumodel(max_len, n_var, rec_units, ndense=[50, 10], l1_reg=0,
              l2_reg=0, rec_act='sigmoid', extra_lab='none', rec_kernel_init='VarianceScaling',
             dense_kernel_init='lecun_uniform'):
    
    hidden = x_in = Input(shape=(max_len, n_var,))
    
    #hidden = Dense(50, kernel_initializer=dense_kernel_init, name='dense_0' )(hidden)
    hidden = Dense(50, kernel_initializer=dense_kernel_init, name='dense_0' )(hidden)
    hidden = Activation('relu', name = 'relu_0')(hidden)
    
    hidden = Dense(10, kernel_initializer=dense_kernel_init, name='dense_1' )(hidden)
    hidden = Activation('relu', name = 'relu_1')(hidden)

    hidden = Dense(3, kernel_initializer=dense_kernel_init, name = 'dense_2')(hidden)
    hidden = Activation('softmax', name = 'output_softmax')(hidden)
    
    model = Model(inputs=x_in, outputs=hidden)
    
    return model

In [16]:
## nongru model
l1_reg = 0
l2_reg = 0

#nogru_model = nongrumodel(15, 6, 120, [50, 10], l1_reg=l1_reg, l2_reg=l2_reg)
nogru_model = nongrumodel(50, 120, 120, [50, 10], l1_reg=l1_reg, l2_reg=l2_reg)
nogru_model.compile(optimizer='adam', loss=['categorical_crossentropy'], metrics=['accuracy'])  

In [17]:
nogru_model.load_weights("./gru_test3/gru_floating.weights.h5", skip_mismatch=True, by_name=True) 

print("nogru model")
nogru_model.summary()

#loading weights by layer to confirm it actually work
for layer in nogru_model.layers:
    #print(layer.name, layer.input_shape)
    
    #weights = layer.get_weights()
    os.makedirs(f'./weights/weights_{layer.name}_weights_biases_pkgs', exist_ok=True)

    # getting types
    #print(type(weights))
    #print(type(layer.get_weights()))

    #print(layer.name, layer.get_weights())
    #for weight_list in layer.get_weights():
    #    for indv_weight in weight_list:
    #        print(str(indv_weight))

    #np.ndarray.tofile(f'./weights/weights_{layer.name}_weights_biases_pkgs/{layer.name}_weights.txt', "")
    #np.savetxt(f'./weights/weights_{layer.name}_weights_biases_pkgs/{layer.name}_weights.txt', weights) 
                                                # errors out since the data shape is inconsistent
    
    with open(f'./weights/weights_{layer.name}_weights_biases_pkgs/{layer.name}_weights.txt', 'w') as f_weight:
        with open(f'./weights/weights_{layer.name}_weights_biases_pkgs/{layer.name}_biases.txt', 'w') as f_bias:
            for weight_list in layer.get_weights():
                for weight_set in weight_list:
                                    #print(type(weight_set))
                    if isinstance(weight_set, np.float32):
                        f_bias.write((str(weight_set))+"\n")
                    else:                    
                        for weight_indv in weight_set:
                                #print(type(weight_indv))
                            f_weight.write((str(weight_indv))+"\n")
                    #if isinstance(weight_set, 'numpy.float32'):
                    #    for weight_indv in weight_set:
                    #        f.write("".join(str(weight_indv))+"\n")
                    #else
                    #    f.write("".join(str(weight_indv))+"\n")
                #f.write(".\n")
            #f.write("\n".join(np.array(weight).to_list()))

# this helper function doesn't work, I am guessing because the size/ dimensions of my weights input were different than caleb's
#helper_functions.dump_weights(nogru_model)
        

nogru model
Model: "model_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_6 (InputLayer)        [(None, 50, 120)]         0         
                                                                 
 dense_0 (Dense)             (None, 50, 50)            6050      
                                                                 
 relu_0 (Activation)         (None, 50, 50)            0         
                                                                 
 dense_1 (Dense)             (None, 50, 10)            510       
                                                                 
 relu_1 (Activation)         (None, 50, 10)            0         
                                                                 
 dense_2 (Dense)             (None, 50, 3)             33        
                                                                 
 output_softmax (Activation)  (None, 50, 3)    

In [19]:
x_test_toptag = np.load('/home/quin/HLS4ML_VS_MANUAL/documents/Benchmarks/Btagging/qkeras/traindata_toptag/x_test.npy')
x_test_gru_inport = np.ascontiguousarray(x_test_toptag)
#print(x_test_gru_inport)

#x_gru_chopped = x_test_gru_inport[:15, :]
#x_gru_chopped = np.resize(x_test_gru_inport, (x_test_gru_inport.shape[0], 15, 6))
x_gru_chopped = np.resize(x_test_gru_inport, (x_test_gru_inport.shape[0], 15, 6)) #x_test_gru_inport.shape[0], 
#print(x_gru_chopped)

# top tag data
y_gruonly = gru_layer_only.predict(x_gru_chopped)
#print(y_gruonly)
y_gruonly_shaped = np.resize(y_gruonly, (x_test_gru_inport.shape[0], 50, 120))
#print(y_gruonly_shaped)
predict_nogru = nogru_model.predict(y_gruonly_shaped)
#print(predict_nogru)

# making 0's
x_test_gru_zeros = np.zeros((x_test_gru_inport.shape[0], 15, 6))
#print(x_test_gru_zeros)

# zeroed input data
y_gruonly_zeros = gru_layer_only.predict(x_test_gru_zeros)
print(y_gruonly_zeros)
y_gruonly_zeros_shaped = np.resize(y_gruonly_zeros, (x_test_gru_inport.shape[0], 50, 120))
print(y_gruonly_zeros_shaped)
predict_nogru_zeroes = nogru_model.predict(y_gruonly_zeros_shaped)

## writing the training data files
os.makedirs(f'./acc/testingData_noGru', exist_ok=True)
acc = (16, 6)
#helper_functions.gen_test(acc, y_gruonly_zeros_shaped, "zeroesData")
#helper_functions.gen_test(acc, y_gruonly_shaped, "toptaggingData")

## writing the predicted outputs files, uncomment out if need to make outputs again 
os.makedirs(f'./acc/no_gru', exist_ok=True)

with open(f'./acc/no_gru/keras_zeros.txt', 'w') as f: #_noSoft
    #f.write(str(y_nogru_zeroes))
    for y_nogru_zeroes_layers in predict_nogru_zeroes:
        for y_nogru_zeroes_individual in y_nogru_zeroes_layers:
                f.write(str(y_nogru_zeroes_individual)+"\n")
    
with open(f'./acc/no_gru/keras_predict.txt', 'w') as f: #_noSoft
    for y_nogru_layers in predict_nogru:
        for y_nogru_individual in y_nogru_layers:
                f.write(str(y_nogru_individual)+"\n")


    #f.write(str(y_nogru))
#np.savetxt(f"./acc/keras_predict.txt", y_nogru), fmt='%.4f')


#y_nogru_model_pred = np.argmax(y_nogru, axis=1)
#y_nogru_handmade = np.argmax(, axis=1)

#print("no gru model Acurracy: {}".format(accuracy_score(np.argmax(y_nogru, axis=1) np.argmax(, axis=1))))

#from sklearn.metrics import classification_report, confusion_matrix

#print (classification_report(y_nogru_model_pred, y_nogru_handmade))



  1/624 [..............................] - ETA: 11s

624/624 [==============================] - 1s 2ms/step
[[-0.06631984  0.58159876  0.21630402 ... -0.31820354 -0.01579527
  -0.29636505]
 [-0.06631984  0.58159876  0.21630402 ... -0.31820354 -0.01579527
  -0.29636505]
 [-0.06631984  0.58159876  0.21630402 ... -0.31820354 -0.01579527
  -0.29636505]
 ...
 [-0.06631984  0.58159876  0.21630402 ... -0.31820354 -0.01579527
  -0.29636505]
 [-0.06631984  0.58159876  0.21630402 ... -0.31820354 -0.01579527
  -0.29636505]
 [-0.06631984  0.58159876  0.21630402 ... -0.31820354 -0.01579527
  -0.29636505]]
[[[-0.06631984  0.58159876  0.21630402 ... -0.31820354 -0.01579527
   -0.29636505]
  [-0.06631984  0.58159876  0.21630402 ... -0.31820354 -0.01579527
   -0.29636505]
  [-0.06631984  0.58159876  0.21630402 ... -0.31820354 -0.01579527
   -0.29636505]
  ...
  [-0.06631984  0.58159876  0.21630402 ... -0.31820354 -0.01579527
   -0.29636505]
  [-0.06631984  0.58159876  0.21630402 ... -0.31820354 -0.01579527
   -0.29636505]
  [-0.06631984  0.58159876  0.21

In [ ]:
## Gen weights
acc = (16, 6)
#helper_functions.gen_weight(acc, nogru_model)

# I would need to keep track of each layer for the gen_weight_txt and change it in helper function,
# the order ends up being different and errors and I can fix it but it creates more problems
#weights_file = "./weights/weights_dense_0_weights_biases_pkgs/dense_0_weights.txt"
#biases_file = "./weights/weights_dense_0_weights_biases_pkgs/dense_0_biases.txt"
#helper_functions.gen_weight_txt(acc)

#error=f"No_gru Predict Script Completed"
#os.system(f'printf "Complete converting data to binary" | mail -s "{error}" ljdono@uw.edu')